In [1]:
import random
import time
import math
import numpy as np

import deap
from deap import base, creator, tools

In [2]:
plantType = ["Null", "Corn", "Beans", "Squash"]

DEFAULT_NITROGEN = 50 # 50 ppm as baseline for the soil.
DEFAULT_LIGHT = 1 # 100% sunlight as baseline for the light level.
DEFAULT_YIELD = 0

CELLDISTANCE = 0

def create_plant(plantTypeVal):
    plantInd = [plantType[plantTypeVal], int(DEFAULT_NITROGEN), float(DEFAULT_LIGHT), float(DEFAULT_YIELD)]
    return plantInd

# Modifier Layout: [Plant-Corn Nitrogen, Plant-Corn Light, Plant-Beans Nitrogen, Plant-Beans Light, Plant-Squash Nitrogen, Plant-Squash Light]
CORN_MODIFIER = [-15, -.2, -10, .1, -5, -.25]
BEANS_MODIFIER = [20, 0, -5, -.1, 15, -.05]
SQUASH_MODIFIER = [5, 0, 5, -.05, -8, -.15]

def apply_plant_modifier(castingPlant, targetPlant):
    if (castingPlant[0] == "Null"):
        return targetPlant

    modifier = []
    if (castingPlant[0] == "Corn"):
        modifier = CORN_MODIFIER
    elif (castingPlant[0] == "Beans"):
        modifier = BEANS_MODIFIER
    elif (castingPlant[0] == "Squash"):
        modifier = SQUASH_MODIFIER

    if (targetPlant[0] == "Corn"):
        targetPlant[1] += modifier[0]
        targetPlant[2] += modifier[1]
    elif (targetPlant[0] == "Beans"):
        targetPlant[1] += modifier[2]
        targetPlant[2] += modifier[3]
    elif (targetPlant[0] == "Squash"):
        targetPlant[1] += modifier[4]
        targetPlant[2] += modifier[5]

    # Clamp nitrogen >= 0 and light to [0, 1]
    targetPlant[1] = max(0, targetPlant[1])
    targetPlant[2] = max(0.0, min(1.0, targetPlant[2]))

    return targetPlant

def calculate_all_modifiers(plantArray):
    for row in range(len(plantArray)):
        for col in range(len(plantArray[row])):
            curPlant = plantArray[row][col]
            if (row != 0):
                apply_plant_modifier(plantArray[row-1][col], curPlant)
                if (col != 0):
                    apply_plant_modifier(plantArray[row-1][col-1], curPlant)
                if (col != plantArray.shape[1]-1):
                    apply_plant_modifier(plantArray[row-1][col+1], curPlant)
            if (row != plantArray.shape[0]-1):
                apply_plant_modifier(plantArray[row+1][col], curPlant)
                if (col != 0):
                    apply_plant_modifier(plantArray[row+1][col-1], curPlant)
                if (col != plantArray.shape[1]-1):
                    apply_plant_modifier(plantArray[row+1][col+1], curPlant)
            if (col != 0):
                apply_plant_modifier(plantArray[row][col-1], curPlant)
            if (col != plantArray.shape[1]-1):
                apply_plant_modifier(plantArray[row][col+1], curPlant)

def calculate_yield_val(value, optMin, optMax):
    mid = (optMin + optMax) / 2
    half_range = (optMax - optMin) / 2
    yieldVal = max(0, half_range - abs(value - mid))
    return yieldVal

CORN_NITROGEN_RANGE = [120, 200]
CORN_LIGHT_RANGE = [.8, 1]
BEANS_NITROGEN_RANGE = [30, 60]
BEANS_LIGHT_RANGE = [.5, .75]
SQUASH_NITROGEN_RANGE = [60, 120]
SQUASH_LIGHT_RANGE = [.65, .9]

def calculate_plant_yield(plantInd):
    if (plantInd[0] == "Null"):
        return plantInd

    nitrogenRange = []
    lightRange = []

    if (plantInd[0] == "Corn"):
        nitrogenRange = CORN_NITROGEN_RANGE
        lightRange = CORN_LIGHT_RANGE
    elif (plantInd[0] == "Beans"):
        nitrogenRange = BEANS_NITROGEN_RANGE
        lightRange = BEANS_LIGHT_RANGE
    elif (plantInd[0] == "Squash"):
        nitrogenRange = SQUASH_NITROGEN_RANGE
        lightRange = SQUASH_LIGHT_RANGE

    nitrogenValue = calculate_yield_val(plantInd[1], nitrogenRange[0], nitrogenRange[1])
    lightValue = calculate_yield_val(plantInd[2], lightRange[0], lightRange[1])

    plantInd[3] = nitrogenValue * lightValue
    return plantInd

def calculate_all_yields(plantArray):
    totalYield = 0
    for row in range(len(plantArray)):
        for col in range(len(plantArray[row])):
            calculate_plant_yield(plantArray[row][col])
            totalYield += plantArray[row][col][3]
    return totalYield

In [3]:
# Test functions

def print_plant_array(plantArray):
    cols = len(plantArray[0])
    sep = "+" + ("-" * 16 + "+") * cols
    for row in plantArray:
        print(sep)
        cells = [f" {p[0]:<6} Y:{p[3]:>5.2f} " for p in row]
        print("|" + "|".join(cells) + "|")
    print(sep)

def print_full_plant_array(plantArray):
    cols = len(plantArray[0])
    sep = "+" + ("-" * 32 + "+") * cols
    for row in plantArray:
        print(sep)
        cells = [f" {p[0]:<6} N:{p[1]:>4}  L:{p[2]:.2f}  Y:{p[3]:>5.2f} " for p in row]
        print("|" + "|".join(cells) + "|")
    print(sep)



def test_yields(xSize, ySize):
    fieldArray = np.array([[create_plant(random.randint(0, 3)) for _ in range(ySize)] for _ in range(xSize)], dtype=object)
    calculate_all_modifiers(fieldArray)
    totalYield = calculate_all_yields(fieldArray)
    print_full_plant_array(fieldArray)
    print(f"Total Yield: {totalYield}")

test_yields(5, 5)
    

+--------------------------------+--------------------------------+--------------------------------+--------------------------------+--------------------------------+
| Squash N:  37  L:0.60  Y: 0.00 | Squash N:  52  L:0.55  Y: 0.00 | Beans  N:  35  L:1.00  Y: 0.00 | Corn   N:  75  L:1.00  Y: 0.00 | Null   N:  50  L:1.00  Y: 0.00 |
+--------------------------------+--------------------------------+--------------------------------+--------------------------------+--------------------------------+
| Null   N:  50  L:1.00  Y: 0.00 | Corn   N:  65  L:0.80  Y: 0.00 | Null   N:  50  L:1.00  Y: 0.00 | Null   N:  50  L:1.00  Y: 0.00 | Squash N:  40  L:0.50  Y: 0.00 |
+--------------------------------+--------------------------------+--------------------------------+--------------------------------+--------------------------------+
| Null   N:  50  L:1.00  Y: 0.00 | Null   N:  50  L:1.00  Y: 0.00 | Corn   N:   0  L:0.20  Y: 0.00 | Corn   N:  35  L:0.60  Y: 0.00 | Null   N:  50  L:1.00  Y: 0.00 

In [ ]:
GRID_X = 5
GRID_Y = 5
N_GENES = GRID_X * GRID_Y

POP_SIZE = 100

def evaluate(individual):
    plantArray = np.array([[create_plant(individual[row * GRID_Y + col]) for col in range(GRID_Y)] for row in range(GRID_X)], dtype=object)
    calculate_all_modifiers(plantArray)
    totalYield = calculate_all_yields(plantArray)
    return (totalYield,)

In [15]:
def run_random():
    bestArray = []
    bestYield = -1
    for i in range(POP_SIZE):
        fieldArray = np.array([[create_plant(random.randint(0, 3)) for _ in range(GRID_Y)] for _ in range(GRID_X)], dtype=object)
        calculate_all_modifiers(fieldArray)
        totalYield = calculate_all_yields(fieldArray)
        if (totalYield > bestYield):
            bestYield = totalYield
            bestArray = fieldArray
    print_full_plant_array(bestArray)
    print(f"Best Yield: {bestYield}")

run_random()

+--------------------------------+--------------------------------+--------------------------------+--------------------------------+--------------------------------+
| Squash N:  72  L:0.75  Y: 1.20 | Beans  N:  60  L:0.75  Y: 0.00 | Squash N:  72  L:0.75  Y: 1.20 | Null   N:  50  L:1.00  Y: 0.00 | Null   N:  50  L:1.00  Y: 0.00 |
+--------------------------------+--------------------------------+--------------------------------+--------------------------------+--------------------------------+
| Beans  N:  55  L:0.80  Y: 0.00 | Squash N:  79  L:0.55  Y: 0.00 | Null   N:  50  L:1.00  Y: 0.00 | Beans  N:  45  L:0.60  Y: 1.50 | Beans  N:  45  L:0.75  Y: 0.00 |
+--------------------------------+--------------------------------+--------------------------------+--------------------------------+--------------------------------+
| Null   N:  50  L:1.00  Y: 0.00 | Null   N:  50  L:1.00  Y: 0.00 | Beans  N:  30  L:0.85  Y: 0.00 | Squash N: 135  L:0.45  Y: 0.00 | Beans  N:  35  L:0.55  Y: 0.25 

In [ ]:
def run_ga():
    if not hasattr(creator, "FitnessMax"):
        creator.create("FitnessMax", base.Fitness, weights=(1.0,))
    if not hasattr(creator, "Individual"):
        creator.create("Individual", list, fitness=creator.FitnessMax)

    toolbox = base.Toolbox()

    toolbox.register("attr_plant", random.randint, 0, 3)
    toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_plant, n=N_GENES)
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)

    toolbox.register("evaluate", evaluate)
    toolbox.register("mate", tools.cxTwoPoint)
    toolbox.register("mutate", tools.mutUniformInt, low=0, up=3, indpb=0.1)
    toolbox.register("select", tools.selTournament, tournsize=3)
    
    POP_SIZE = 100
    N_GEN = 100
    CROSSOVER = 0.7
    MUTATION = 0.2

    pop = toolbox.population(n=POP_SIZE)
    for ind in pop:
        ind.fitness.values = toolbox.evaluate(ind)

    hof = tools.HallOfFame(1)
    hof.update(pop)

    for gen in range(1, N_GEN + 1):
        offspring = toolbox.select(pop, len(pop))
        offspring = list(map(toolbox.clone, offspring))

        # Crossover
        for c1, c2 in zip(offspring[::2], offspring[1::2]):
            if random.random() < CROSSOVER:
                toolbox.mate(c1, c2)
                del c1.fitness.values
                del c2.fitness.values

        # Mutation
        for mutant in offspring:
            if random.random() < MUTATION:
                toolbox.mutate(mutant)
                del mutant.fitness.values

        invalid = [ind for ind in offspring if not ind.fitness.valid]
        for ind in invalid:
            ind.fitness.values = toolbox.evaluate(ind)

        pop[:] = offspring

        hof.update(pop)

        fitness = [ind.fitness.values[0] for ind in pop]
        print(f"{gen:>4} {np.mean(fitness):>10.4f} {np.max(fitness):>10.4f}")

    print(f"\nBest Fitness: {hof[0].fitness.values[0]:.4f}")
    best = hof[0]
    bestArray = np.array([[create_plant(best[row * GRID_Y + col]) for col in range(GRID_Y)] for row in range(GRID_X)], dtype=object)
    calculate_all_modifiers(bestArray)
    bestYield = calculate_all_yields(bestArray)
    print_full_plant_array(bestArray)
    print(f"Best Yield: {bestYield}")

    return pop, hof

pop, hof = run_ga()

# continue from here

   1     1.8950     8.1500
   2     2.6210     8.1500
   3     3.0750     7.4500
   4     3.5820     8.4500
   5     4.1500     8.9500
   6     5.6820     9.6000
   7     6.6055    11.9500
   8     7.9195    13.7000
   9     8.6380    13.7000
  10     9.3070    13.7000
  11    10.2365    14.3500
  12    10.7185    14.4500
  13    11.4355    15.4500
  14    12.2960    15.4500
  15    13.1820    15.4500
  16    13.5385    15.4500
  17    13.6370    15.5500
  18    14.6235    15.6000
  19    14.5360    15.6000
  20    14.4260    15.6000
  21    14.6225    15.6000
  22    14.4795    15.6000
  23    14.8535    15.6000
  24    14.8130    15.6000
  25    14.3560    15.6000
  26    15.1040    15.6000
  27    14.9035    15.6000
  28    14.6740    15.6000
  29    14.7290    15.6000
  30    14.6425    15.6000
  31    14.9005    15.6000
  32    14.5055    15.6000
  33    15.2480    15.6000
  34    14.7725    15.6000
  35    14.9000    15.6000
  36    14.7290    15.6000
  37    14.5480    15.6000
 